# AVENIQ SENTINEL // PHASE 3.1: CLOUD DATASET PROFILING
## Official CIC-IDS2017 Dataset Profiling & Schema Reconciliation

**Product:** Aveniq Sentinel by Aveniq  
**Target Repository:** https://github.com/sparxweb/aveniq-sentinel  
**Environment Target:** Kaggle Notebooks (Primary Cloud) / Google Colab (Backup)  
**Compute Policy:** Free CPU / zero GPU requirement / resource-conscious chunking  

---

In [ ]:
import sys
import platform
from pathlib import Path

# Path resolution for local vs cloud execution
PROJECT_ROOT = Path('.').resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import sklearn
import pyarrow

print('=== CLOUD ENVIRONMENT & LIBRARY TELEMETRY ===')
print(f'Python: {platform.python_version()}')
print(f'Pandas: {pd.__version__}')
print(f'NumPy: {np.__version__}')
print(f'Scikit-learn: {sklearn.__version__}')
print(f'PyArrow: {pyarrow.__version__}')

=== CLOUD ENVIRONMENT & LIBRARY TELEMETRY ===
Python: 3.11.16
Pandas: 2.3.3
NumPy: 1.26.4
Scikit-learn: 1.5.2
PyArrow: 17.0.0


In [ ]:
from neuralshield.ml.profiler import DatasetProfiler, CICIDS2017_FILE_METADATA
from config.settings import SAMPLE_DATA_DIR, RAW_DATA_DIR

# Cloud input discovery (Kaggle mount vs local repo)
kaggle_input = Path('/kaggle/input/cicids2017')
if kaggle_input.exists():
    active_data_dir = kaggle_input
    data_source = 'Kaggle Cloud Mount (/kaggle/input/cicids2017)'
elif RAW_DATA_DIR.exists() and list(RAW_DATA_DIR.glob('*.csv')):
    active_data_dir = RAW_DATA_DIR
    data_source = f'Local Benchmark CSV Directory ({RAW_DATA_DIR})'
else:
    active_data_dir = SAMPLE_DATA_DIR
    data_source = f'Deterministic Sandbox Sample ({SAMPLE_DATA_DIR})'

print(f'Active Data Source: {data_source}')
profiler = DatasetProfiler(active_data_dir)

Active Data Source: Deterministic Sandbox Sample (C:\Users\HP\OneDrive\Desktop\NEURALSHIELD AI\data\sample)


In [ ]:
print('=== OFFICIAL CIC-IDS2017 8-FILE BENCHMARK METADATA ===')
meta_rows = []
for fname, meta in CICIDS2017_FILE_METADATA.items():
    meta_rows.append({
        'File Name': fname,
        'Day': meta['day'],
        'Date': meta['date'],
        'Expected Flows': f"{meta['expected_flows']:,}",
        'Size (MB)': meta['approx_size_mb'],
        'Traffic / Attack Types': meta['traffic_type'],
    })
meta_df = pd.DataFrame(meta_rows)
meta_df

=== OFFICIAL CIC-IDS2017 8-FILE BENCHMARK METADATA ===


In [ ]:
# Load active data frame for live profiling
sample_parquet = SAMPLE_DATA_DIR / 'cic_ids2017_sample.parquet'
sample_csv = SAMPLE_DATA_DIR / 'cic_ids2017_sample.csv'

if sample_parquet.exists():
    df = pd.read_parquet(sample_parquet)
elif sample_csv.exists():
    df = pd.read_csv(sample_csv)
else:
    raise FileNotFoundError('No dataset found for profiling.')

profile = profiler.inspect_dataframe_profile(df, source_name='Active Baseline Telemetry')
print(f'Flow Records: {profile["row_count"]:,}')
print(f'Feature Columns: {profile["column_count"]}')
print(f'Resident Memory: {profile["memory_mb"]} MB')
print(f'Null Cells: {profile["total_null_cells"]}')
print(f'Infinite Value Cells: {profile["total_infinite_cells"]}')
print(f'Exact Duplicate Rows: {profile["duplicate_rows"]} ({profile["duplicate_percentage"]}%)')
print(f'Binary Distribution: {profile["binary_distribution"]}')

Flow Records: 1,505
Feature Columns: 39
Resident Memory: 0.45 MB
Null Cells: 1
Infinite Value Cells: 4
Exact Duplicate Rows: 5 (0.33%)
Binary Distribution: {'Benign': 1193, 'Attack': 312}
